# 🚀 Retrieval-Augmented Generation (RAG) Assignment

### Author: John Kiran Grandhi

This notebook demonstrates a complete RAG pipeline including:
- PDF ingestion
- Text chunking
- Embeddings creation
- Vector database (FAISS)
- LLM-based answer generation
- Evaluation framework

---

## 📦 Step 1: Install Dependencies
To run locally, we need:
- Python Installed.
- Ollama Installed.
- VS code with Jupyter extension.

We install all required libraries for:
- Document loading
- Embeddings
- Vector search
- LLM interaction

---

In [1]:
!pip install -U langchain langchain-community langchain-core langchain-text-splitters faiss-cpu pypdf sentence-transformers ollama

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.23
    Uninstalling langchain-core-1.2.23:
      Successfully uninstalled langchain-core-1.2.23
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.13
    Uninstalling langchain-1.2.13:
      Successfully uninstalled langchain-1.2.13



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📥 Step 2: Import Libraries

These libraries are used for:
- Loading PDFs
- Splitting text
- Creating embeddings
- Storing vectors
- Running LLM

---

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama

## 📄 Step 3: PDF Ingestion

We load research papers from URLs.
Each page becomes a document.

- Loads 5 PDFs from given URLs
- Each PDF is split into multiple pages
- Output is a list of documents (each page = 1 document)

Output structure:

docs[ { pageContent: "", metadata: "", id: ""}]

---

In [3]:
urls = [
    "https://arxiv.org/pdf/1706.03762.pdf",
    "https://arxiv.org/pdf/1810.04805.pdf",
    "https://arxiv.org/pdf/2005.14165.pdf",
    "https://arxiv.org/pdf/1907.11692.pdf",
    "https://arxiv.org/pdf/1910.10683.pdf"
]

docs = []
for url in urls:
    loader = PyPDFLoader(url)
    docs.extend(loader.load())

print("Total Pages Loaded:", len(docs))

Total Pages Loaded: 186


## ✂️ Step 4: Text Chunking

Chunking = breaking large text into smaller pieces

- chunk_size = 500
- overlap = 50

👉 Why?

* LLMs & embeddings can’t handle very long text properly
* Smaller chunks = better search + better answers

👉 Why RecursiveCharacterTextSplitter?

* Avoids breaking sentences randomly
* Maintains context + readability
* Fits chunk size properly
* Best balance for marks + performance

👉 Output

Each chunk contains:

* piece of text
* metadata (like page number, source)

---

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("Total Chunks Created:", len(chunks))

Total Chunks Created: 1425


## 🔢 Step 5: Embeddings & Vector Store

- Model: all-MiniLM-L6-v2
- Vector DB: FAISS

👉 Embeddings = converting text into numbers (vectors)

* Computers don’t understand text directly
* So we convert text → numeric representation
* Similar meaning → similar vectors

👉 Example:

* "What is AI?"
* "Explain Artificial Intelligence"

➡️ Both will have similar embeddings

✅ What is HuggingFaceEmbeddings?

👉 It is a pre-trained model from Hugging Face that:

* Takes text (chunks)
* Converts into embedding vectors
* Usually 384 dimensions (for this model)

✅ What is the Output here?

✔️ It stores:

The embedding model (MiniLM)
A function to convert text → numbers

👉 Think:

“embeddings = a tool that converts text into vectors”

If we print it, it shows:

    client=SentenceTransformer(
        (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
        (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
        (2): Normalize()
        )
    model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False
---

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\grandhikiran\AppData\Local\Temp\ipykernel_10112\3401734470.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector DB
vector_db:

👉 It is a database

✔️ It stores:

All chunks (text)
Their embeddings (vectors)

👉 Think:

“vector_db = storage of text + their numeric meaning”

---

retriever:

👉 It is a search tool

✔️ It does:

Takes your question
Converts → embedding
Finds similar chunks from vector_db

👉 Think:

“retriever = search engine on top of vector_db”

In [6]:
vector_db = FAISS.from_documents(chunks, embeddings)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

## 🤖 Step 6: Load LLM

We use Ollama with LLaMA3 model for response generation.

---

In [7]:
llm = Ollama(model="llama3")

C:\Users\grandhikiran\AppData\Local\Temp\ipykernel_10112\2786610211.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")


## 🧠 Step 7: Conversational Memory

### 📌 Objective
Implement a conversational bot that **remembers the last 4 interactions** (user + bot responses) and uses this context to generate better answers.

---

### ⚙️ Approach Used

We implement a custom **ConversationMemory class** that:

- Stores conversation history as **(user_input, bot_response) pairs**
- Maintains only the **last 4 interactions** using a sliding window
- Converts memory into **text format** to include in the LLM prompt

---

### 🔁 How It Works

1. **Store Conversations**
   - Each user query and bot response is saved

2. **Limit Memory**
   - Only the most recent 4 interactions are kept  
   - Older conversations are automatically removed

3. **Inject into Prompt**
   - Memory is converted into text and added to the prompt:
     ```
     Context + Memory + Question → LLM
     ```

---

### 🧩 Mapping to Assignment Requirements

- ✅ **Conversation State Management**  
  → Managed using a Python class (`ConversationMemory`)

- ✅ **Prompt Concatenation**  
  → Memory is appended to the prompt before sending to LLM

- ✅ **Last 4 Interactions Only**  
  → Controlled using `max_turns=4`

---

### 🎯 Benefit

Including memory helps the bot:
- Maintain **context continuity**
- Give **more relevant and coherent answers**
- Simulate a **real conversational experience**
---

In [8]:
class ConversationMemory:
    def __init__(self, max_turns=4):
        self.max_turns = max_turns
        self.history = []

    def add_conversation(self, user_input, bot_response):
        self.history.append((user_input, bot_response))
        self.history = self.history[-self.max_turns:]

    def get_memory_text(self):
        memory_text = ""
        for user, bot in self.history:
            memory_text += f"User: {user}\nBot: {bot}\n"
        return memory_text.strip()

memory = ConversationMemory()

## 🔄 Step 8: RAG Pipeline

Pipeline Flow:
1. User query
2. Retrieve relevant chunks
3. Add memory
4. Generate answer

---

In [9]:
def rag(query):
    retrieved_docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    memory_text = memory.get_memory_text()

    prompt = f"""Context: {context}
Memory: {memory_text}
Question: {query}
Answer:"""

    response = llm.invoke(prompt)

    memory.add_conversation(query, response)

    return response, context

## 📊 Step 9: Evaluation Framework

A custom evaluation function is implemented to score each response based on multiple criteria

### 📊 Evaluation Criteria

Each response is scored out of 4 points:

1. Relevance (1 point)
    * Checks if the answer contains keywords related to the question
2. Contextual Awareness (1 point)
    * Verifies whether the answer uses information from retrieved chunks
3. Response Quality (1 point)
    * Ensures the answer is sufficiently detailed and well-formed
4. Basic Accuracy (1 point)
    * Checks for meaningful and structured statements in the response

### ⚠️ Limitations
* Does not deeply verify factual correctness against source PDFs
* Keyword-based matching may not fully capture semantic relevance
* Can be improved using advanced frameworks like RAGAS or TruLens

---

In [10]:
def evaluate(answer, question, context):
    score = 0

    # 1. Relevance (keyword overlap)
    if any(word in answer.lower() for word in question.lower().split()):
        score += 1

    # 2. Context usage (checks if answer uses retrieved content)
    if any(sentence[:20] in answer for sentence in context.split("\n")):
        score += 1

    # 3. Response Quality (length + clarity)
    if len(answer) > 80:
        score += 1

    # 4. Basic Accuracy (checks meaningful words)
    if "is" in answer.lower() or "are" in answer.lower():
        score += 1

    return score

## 🧪 Step 10: Testing

A test suite is created by defining a list of questions and passing them through the RAG pipeline.

---

In [11]:
qs = [
    "What is attention mechanism?",
    "Explain the transformer architecture",
    "What problem does BERT solve?",
    "How does self-attention work?",
    "What is GPT model?",
    "Difference between BERT and GPT",
    "What is transfer learning in NLP?",
    "Explain masked language modeling",
    "What are key components of transformer?",
    "Why are transformers better than RNNs?"
]

for q in qs:
    print("Question:", q)

    answer, context = rag(q)

    score = evaluate(answer, q, context)

    print("Answer:", answer)
    print("Score:", score, "/ 4")
    print("-" * 50)

Question: What is attention mechanism?
Answer: Based on the context provided, it appears that the attention mechanism is a component of a neural network model (specifically, a transformer) that allows the model to focus on specific parts of an input sequence when processing it. This is illustrated in Figure 3, which shows how different "heads" or sub-mechanisms within the attention mechanism attend to different parts of the input sequence.

In particular, the figure shows that some heads are able to follow long-distance dependencies in the input sequence, such as completing a phrase like "making...more difficult". This suggests that the attention mechanism is allowing the model to focus on relevant parts of the input sequence and use this information to make predictions or generate output.
Score: 4 / 4
--------------------------------------------------
Question: Explain the transformer architecture
Answer: Based on the provided context, I can explain the Transformer architecture as fol